# Milestone 3 lab, notebook 3 of 8: A gap is not a zero

## Where we are

| Notebook | What you do |
|---|---|
| 1. A price has a start and an end | products. See one forgotten date count every sale twice |
| 2. Six rows, wrong everywhere | stores. Find out why the weakest check is the best one here |
| **3. A gap is not a zero** (you are here) | stock_counts. Look for a row that is not there |
| 4. The same person twice | members. Find one person behind two keys, and keep their data private |
| 5. A file you do not control | supplier_prices. A supplier renames a column, and nothing fails |
| 6. Same shape, new meaning | supplier_prices. Every number is valid, and 100 times too high |
| 7. A contract for the supplier | Write down what a file must look like, and check it at the door |
| 8. Cards side by side | Compare the seven cards. Then see what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import (
    keep, lab_copy, money, on_lab, plant, planted_case, prepare_week, production, show_card, sql,
    standard_checks,
)

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 8 clean tables.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: Eighteen bottles

**The replenishment team** orders tomorrow's stock every night. Their rule is simple: keep two days of sales on the shelf. So the order is twice what sold today, less what is on the shelf now.

On Friday 28 August, store `S05` sold a lot of milk. Here is what the pipeline told the team to order for Saturday.

In [2]:
sql("""
    SELECT for_date, store_id, product_name, sold_yesterday, stock_counted, order_qty, order_cost_cents
    FROM output.replenishment_list
    WHERE store_id = 'S05' AND product_id = 'P-001'
      AND for_date BETWEEN DATE '2026-08-28' AND DATE '2026-08-30'
    ORDER BY for_date
""")

,for_date,store_id,product_name,sold_yesterday,stock_counted,order_qty,order_cost_cents
0,2026-08-28,S05,Whole milk 2L,6,2,10,1760
1,2026-08-29,S05,Whole milk 2L,9,0,18,3168
2,2026-08-30,S05,Whole milk 2L,4,14,0,0


Look at the middle row, the order for Saturday 29 August. Nine bottles sold. Stock counted: **0**. So the order is 18.

An empty milk shelf on a busy Friday is possible. Is it true? Look at the counts themselves.

In [3]:
sql("""
    SELECT count_date, counted_qty, counted_by, counted_at
    FROM clean.stock_counts
    WHERE store_id = 'S05' AND product_id = 'P-001'
      AND count_date BETWEEN DATE '2026-08-26' AND DATE '2026-08-30'
    ORDER BY count_date
""")

,count_date,counted_qty,counted_by,counted_at
0,2026-08-26,0,scanner,2026-08-26 15:15:00
1,2026-08-27,2,scanner,2026-08-27 19:15:00
2,2026-08-29,14,scanner,2026-08-29 19:15:00
3,2026-08-30,8,scanner,2026-08-30 19:15:00


Count the rows. Five days were asked for, and four came back. **There is no row for 28 August.**

Nobody counted the milk that evening. The shelf was not empty. The pipeline does not know what was on it.

## Step 2: A gap is not a zero

So where did the 0 come from? From one word in the pipeline's SQL.

In [4]:
from lotus_pipeline import publish

print([line.strip() for line in publish.REPLENISHMENT_LIST.splitlines() if "AS stock_counted" in line][0])

coalesce(stock.counted_qty, 0)   AS stock_counted,


`coalesce(..., 0)` means: when there is no count, use zero. It turns "we do not know" into "the shelf is empty".

The answer key knows what was really on the shelf.

In [5]:
case = planted_case(12)
print(f"Sold on Friday:          {case['sold']:g} bottles")
print(f"Really on the shelf:     {case['true_stock']:g}")
print(f"The right order:         {case['true_order']:g}")
print(f"The pipeline's order:    {case['order_with_no_count']:g}")
print(f"Ordered too much:        {case['over_order']:g} bottles, {money(case['over_order'] * 176)} at $1.76 each")

Sold on Friday:          9 bottles
Really on the shelf:     3
The right order:         15
The pipeline's order:    18
Ordered too much:        3 bottles, $5.28 at $1.76 each


Three bottles too many, $5.28. Tonight it is small, because only one product was skipped.

Think about the size it can have. If the scanner fails and a whole store is not counted, every product is ordered as if every shelf were empty. In a supermarket that is fresh food in the bin.

This table has a name for its kind. It is a **snapshot table**: it holds the state of something on each day. A sales line is the opposite: it records a thing that happened.

| | A table of things that happened | A snapshot table |
|---|---|---|
| Example | `sales_lines` | `stock_counts` |
| One row | One sale | The shelf, on one day |
| No row for a day | Nothing was sold. A true zero | Nobody looked. **A gap** |

In [6]:
show_card("stock_counts", 1)

**From the check card `checks/stock_counts.md`, question 1. What does one row mean?**

One product in one store on one day: how much was on the shelf at closing time.

The key is `count_date` + `store_id` + `product_id`.

This table is a **snapshot**. It does not hold things that happened, like a sale. It holds the state of something on each day. So a day that is missing is a **gap**. It is not a zero.

## Step 3: Why no check saw it

Try the checks you already know. First, the cell must be filled.

In [7]:
sql("SELECT count(*) AS rows, count(*) - count(counted_qty) AS empty_counts FROM clean.stock_counts")

,rows,empty_counts
0,8276,0


In [8]:
standard_checks()

1. The table is not empty           0 bad rows  PASS
2. The key is not null              0 bad rows  PASS
3. The key is unique                0 bad rows  PASS
4. The status is an allowed value   0 bad rows  PASS
5. The total is positive            0 bad rows  PASS


No empty cell in 8,276 rows. Five green checks.

Here is the reason, and it is the lesson of this notebook. **A check is a query that returns the bad rows. It can only return rows that exist.** A count that was never made is not a bad row. It is no row.

To find a row that is missing, you must first build the rows you **expected**.

## Step 4: Build what you expected

Which counts should a day have? One for every product on sale, in every open store.

Count them for `S05`, day by day, beside what really arrived. A product is on sale on a day when that day is inside its validity dates: notebook 1.

In [9]:
sql("""
    SELECT d.day,
           count(*) AS counts_expected,
           count(c.product_id) AS counts_found
    FROM (SELECT DISTINCT business_date AS day FROM clean.till_totals WHERE store_id = 'S05') d
    JOIN clean.products p
      ON d.day BETWEEN p.valid_from AND coalesce(p.valid_to, DATE '9999-12-31')
    LEFT JOIN clean.stock_counts c
      ON c.store_id = 'S05' AND c.product_id = p.product_id AND c.count_date = d.day
    WHERE d.day >= DATE '2026-08-25'
    GROUP BY d.day
    ORDER BY d.day
""")

,day,counts_expected,counts_found
0,2026-08-25,39,39
1,2026-08-26,39,39
2,2026-08-27,39,39
3,2026-08-28,39,38
4,2026-08-29,39,39
5,2026-08-30,39,39


39 expected, and on 28 August 38 found.

Now for every store. The list of days comes from the tills: a store that closed its tills was open. Keep only the expected counts that have no row.

In [10]:
EXPECTED_AND_MISSING = """
    SELECT s.store_id, d.day AS sales_date, p.product_id, p.product_name
    FROM clean.stores s
    CROSS JOIN (SELECT DISTINCT business_date AS day FROM clean.till_totals) d
    JOIN clean.products p
      ON d.day BETWEEN p.valid_from AND coalesce(p.valid_to, DATE '9999-12-31')
    LEFT JOIN clean.stock_counts c
      ON c.store_id = s.store_id AND c.product_id = p.product_id AND c.count_date = d.day
    WHERE c.product_id IS NULL
"""
missing = sql(EXPECTED_AND_MISSING)
print(len(missing), "counts are missing")
missing.groupby(["store_id", "sales_date"]).size().reset_index(name="missing_counts")

40 counts are missing


,store_id,sales_date,missing_counts
0,S01,2026-08-27,39
1,S05,2026-08-28,1


40 rows, not one. The milk at `S05` is there. And 39 more: every product of store `S01` on Thursday 27 August.

Is that a second fault? No. `S01` was closed that day for a planned refit. Nobody was there to count, and that is correct. You met this in milestone 2: head office keeps a calendar, and it knows.

In [11]:
sql("SELECT calendar_date, store_id, kind, note FROM clean.calendar WHERE kind = 'closed'")

,calendar_date,store_id,kind,note
0,2026-08-27,S01,closed,Planned refit


A closed store is not a gap. Take the days out that the calendar says a store was closed.

In [12]:
A_COUNT_FOR_EVERY_DAY = EXPECTED_AND_MISSING + """
      AND NOT EXISTS (
          SELECT 1 FROM clean.calendar cal
          WHERE cal.store_id = s.store_id AND cal.calendar_date = d.day AND cal.kind = 'closed'
      )
"""
yours = sql(A_COUNT_FOR_EVERY_DAY)
yours

,store_id,sales_date,product_id,product_name
0,S05,2026-08-28,P-001,Whole milk 2L


In [13]:
keep("a_count_for_every_day", yours)

Your query:           1 bad rows
The production check: 1 bad rows  (A snapshot has a row for every day)
The same rows.


One row: store `S05`, 28 August, Whole milk 2L. It is the only check so far that returns a row for something that is **not in the table**.

## Step 5: The other two checks on the card

A row that is there can still be wrong. Plant two such faults in a private copy: a product counted twice, and a count below zero.

In [14]:
lab_copy("clean.stock_counts")
plant("""
    INSERT INTO lab.stock_counts SELECT * FROM lab.stock_counts
    WHERE store_id = 'S02' AND product_id = 'P-024' AND count_date = DATE '2026-08-20'
""")
plant("""
    UPDATE lab.stock_counts SET counted_qty = -3
    WHERE store_id = 'S03' AND product_id = 'P-024' AND count_date = DATE '2026-08-20'
""")
for key in ("one_count_per_day", "count_values_allowed", "a_count_for_every_day"):
    found = production(key, on_the_lab=True)
    print(f"{key:<24} {len(found)} bad rows  {sorted(set(found['store_id']))}")

lab.stock_counts is a fresh copy of clean.stock_counts: 8,276 rows.
Planted in the lab copy.
Planted in the lab copy.
one_count_per_day        1 bad rows  ['S02']
count_values_allowed     1 bad rows  ['S03']


a_count_for_every_day    1 bad rows  ['S05']


Each check sees its own fault. And the "row for every day" check still returns only the milk: a count that is there twice, or wrong, is not missing.

One worry on the card gets no check.

In [15]:
show_card("stock_counts", 4)

**From the check card `checks/stock_counts.md`, question 4. Which of those would hurt most?**

| Rank | Worry | Why it is here |
|---|---|---|
| 1 | A product is not counted | The pipeline reads the gap as zero and over-orders. Planted case 12 |
| 2 | A product is counted twice | Two numbers for one shelf. The join doubles the row |
| 3 | A count that cannot be true | Below zero, or half a bottle |
| 4 | A count that is a little off | People miscount. It hurts little, and nothing can test it. **No check** |

People miscount by a bottle or two. It hurts little, and nothing in the data can test it. So the card says: no check.

## What you now know

- A **snapshot table** holds the state of something on each day. A missing day is a gap, not a zero.
- The pipeline read the gap as an empty shelf and ordered 18 bottles where 15 was right.
- A check can only return rows that exist. To find a missing row, build the rows you expected, then look for the ones that are not there.
- A closed store is not a gap. The calendar tells them apart.

**Say it in an interview:** "Stock counts were a snapshot table, so a missing day was a gap, not a zero. Our pipeline used coalesce to zero, read an uncounted shelf as empty, and over-ordered. No check could see it, because a row that is not there cannot fail. I built the rows I expected, every product on sale in every open store, and returned the ones with no count."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "What is good data?", the question "Complete?".

Next: [notebook 4](04_the_same_person_twice.ipynb). A table about people. The same person is in it twice, and both keys are unique.